In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_parquet("../data/cars_clean.parquet")

In [4]:
train, test = train_test_split(df, test_size=0.2, random_state = 50)
print(train.shape, test.shape)

(599286, 20) (149822, 20)


In [5]:
def group_median(train, test, keys):
    med = train.groupby(keys, observed=True)["price"].median().rename("pred")
    return test.join(med, on=keys)["pred"]

pred = group_median(train, test, ["manufacturer", "model", "year"])
pred = pred.fillna(group_median(train, test, ["manufacturer", "model"]))
pred = pred.fillna(group_median(train, test, ["manufacturer"]))
pred = pred.fillna(train["price"].median())

In [8]:
err = (pred - test["price"]).abs()
print(f"MAE: {err.mean()}")
print(f"MAPE: {(err / test["price"]).mean():.1%}")
print(f"Median abs error: {err.median()}")

MAE: 3086.128389021639
MAPE: 12.2%
Median abs error: 2004.5
